In [1]:
from gae import GAE, graph_embed
from utils import GraphData, obtain_ID, convert_label_integer
import numpy as np
import tensorflow as tf
import os 

# Load Malware Graphs
- There is only one processed CFG in data/examples/outputs/cfg to reduce time required by running the code, but in practice, you can have as many as processed CFGs stored in the path and our code is able to load them all.

In [2]:
path =  '../data/examples/outputs/cfg_node_embeddings'
malware = GraphData(path) 
#convert one hot encoding to integer
malware = convert_label_integer(malware)

# Training a graph autoencoder to generate graph embedding (Section III-A1 in the main paper)
- After training the graph autoencoder on graph $G = (X, A)$, we derive the graph representation $Z$ from the encoder: $Z = GCN (X, A)$, and then retrain the graph autoencoder from scratch for the next graph to get its graph representation.
- To simplify the amount of work for testing, we demonstrate with just five processed graphs, but the code is designed to process many graphs stored in the path.

In [3]:
# Getting the list of asm_id when loading the CFGs from the path
# When we obtain the embedding for each graph, we will save the feature with its asm_id and the orginal label

file_list = os.listdir(path)
file_list_x_y = list(filter(lambda x: '_sparse_matrix' not in x and '.npz' in x, file_list))
        

obtain_id= obtain_ID(path, file_list_x_y) 
file_list = obtain_id.read()
file_list = [x.split("_")[2].split(".")[0] for x in file_list]


# This is where the graph embedding Z is stored 
# The next step is to run the concesus clustering on these graph embeddings to find the new cluster labels
# Please go to concensus_clusering.ipynb  for the next step
store_path = '../data/examples/outputs/graph_embedding'


i = 0
fail = 0
for g in malware:
    save_path = 'graph_{}_feature'.format(i)
    filename = os.path.join(store_path, save_path)
    try:
        feature = tf.reshape(graph_embed(g), [-1])
        success = True
        asm_id  = file_list[i]
    except Exception as e: 
        feature = []
        success = False
        asm_id = []
        fail +=1
        
    np.savez(filename, feature = feature, success =success, asm_id = asm_id, y = g.y)
    print("processed graph {}, status is {}".format(i,success))
    i+=1
    
print("Complete! There are {} failed graphs".format(fail))


2024-09-25 12:56:55.770748: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-09-25 12:56:55.801165: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-09-25 12:56:55.802106: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-09-25 12:56:55.812801: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

AUC: 66.6, AP: 65.2
processed graph 0, status is True
AUC: 45.0, AP: 49.7
processed graph 1, status is True
AUC: 44.4, AP: 53.3
processed graph 2, status is True
AUC: 57.8, AP: 64.5
processed graph 3, status is True
AUC: 70.6, AP: 68.2
processed graph 4, status is True
AUC: 65.9, AP: 68.2
processed graph 5, status is True
Complete! There are 0 failed graphs
